# Test Operations Within Mars

Mars implements a lot of operations on the different MarS objects. This notebook test them all

In [1]:
%load_ext autoreload
%autoreload 2
import typing as tp

import sys
import dataclasses
import os
import math
from importlib import reload

import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import mars
from mars import spin_model, spectra_manager, constants, population, concat, serialization

from mars.serialization.serialization import (
    SpinSystemInteractions, SpinSystemMetaData,
    SpinSystemStrains, SampleMetaData, SerializedMarsSession,
    SerializedSampleWidth, SerializedSample, SerializedSpinSystem,
    TimeParameters, PolarizationParameters, ExperimentalParameters
)

from mars.serialization.graph_representation import (
    GraphSample, GraphSpinSystem
)

In [2]:
import torch
import numpy as np
import mars.population.transform as transform
from mars.population.contexts import Context, SummedContext
from mars import spin_model

import torch
import numpy as np
import mars.population.transform as transform
from mars.population.contexts import Context, SummedContext
from mars import spin_model

### 1. Create objects

In [3]:
def make_spin_system_metadata(
    num_electrons: int,
    num_nuclei: int,
    electron_spins: tp.Optional[tp.List[float]] = None,
    nucleus_labels: tp.Optional[tp.List[str]] = None
) -> "SpinSystemMetaData":
    """Creates a batch-ready SpinSystemMetaData with auto-generated interaction pairs."""
    if electron_spins is None:
        electron_spins = [0.5] * num_electrons
        
    if nucleus_labels is None:
        labels_pool = ["14N", "1H", "13C", "31P"]
        nucleus_labels = [labels_pool[i % len(labels_pool)] for i in range(num_nuclei)]

    # Auto-generate valid interaction pairs based on particle counts
    en_pairs = [(i, i % num_nuclei) for i in range(min(num_electrons, num_nuclei))] if num_nuclei > 0 else []
    zfs_pairs = [(i, i) for i in range(num_electrons)]
    dip_pairs = [(0, 1)] if num_electrons > 1 else []
    ee_pairs = zfs_pairs + dip_pairs
    nn_pairs = [(i, i+1) for i in range(num_nuclei - 1)] if num_nuclei > 1 else []
    
    return SpinSystemMetaData(
        electron_spins=electron_spins,
        nucleus_labels=nucleus_labels,
        electron_nucleus_pairs=en_pairs,
        electron_electron_pairs=ee_pairs,
        zfs_pairs=zfs_pairs,
        exchange_dipolar_pairs=dip_pairs,
        nucleus_nucleus_pairs = nn_pairs,
    )

def make_spin_system_interactions(
    batch_size: int,
    num_electrons: int,
    num_hfc: int,
    num_zfs: int,
    num_dip: int,
    num_nn: int,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "SpinSystemInteractions":
    """Creates batched interaction tensors."""
    if num_zfs > 0:
        zfs_components = torch.randn(batch_size, num_zfs, 3, device=device, dtype=dtype) 
        zfs_components = zfs_components - zfs_components.mean(dim=-1, keepdim=True)
    else:
        zfs_components = None
        
    if num_dip > 0:
        dipolar_components = torch.randn(batch_size, num_dip, 3, device=device, dtype=dtype) 
        dipolar_components = dipolar_components - dipolar_components.mean(dim=-1, keepdim=True)
    else:
        dipolar_components = None
    
    return SpinSystemInteractions(
        g_tensor_components=torch.randn(batch_size, num_electrons, 3, device=device, dtype=dtype),
        g_tensor_orientations=torch.randn(batch_size, num_electrons, 3, device=device, dtype=dtype),
        hyperfine_coupling_components=torch.randn(batch_size, num_hfc, 3, device=device, dtype=dtype) if num_hfc > 0 else None,
        hyperfine_coupling_orientations=torch.randn(batch_size, num_hfc, 3, device=device, dtype=dtype) if num_hfc > 0 else None,
        zfs_components=zfs_components,
        dipolar_components=dipolar_components,
        electron_electron_orientations=torch.randn(batch_size, num_zfs + num_dip, 3, device=device, dtype=dtype) if (num_zfs + num_dip) > 0 else None,
        nuclear_coupling_components=torch.randn(batch_size, num_nn, 3, device=device, dtype=dtype) if num_nn > 0 else None,
        nuclear_coupling_orientations=torch.randn(batch_size, num_nn, 3,  device=device, dtype=dtype) if num_nn > 0 else None
    )

def make_spin_system_strains(
    batch_size: int,
    num_electrons: int,
    num_hfc: int,
    num_zfs: int,
    num_dip: int,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "SpinSystemStrains":
    """Creates batched strain tensors."""
    strain_dim = 3
    return SpinSystemStrains(
        g_tensor_strain=torch.randn(batch_size, num_electrons, 3, device=device, dtype=dtype),
        hyperfine_coupling_strain=torch.randn(batch_size, num_hfc, 3, device=device, dtype=dtype) if num_hfc > 0 else None,
        zfs_strain=torch.randn(batch_size, num_zfs, 2, device=device, dtype=dtype) if num_zfs > 0 else None,
        dipolar_strain=torch.randn(batch_size, num_dip, 2, device=device, dtype=dtype) if num_dip > 0 else None
    )

def make_serialized_spin_system(
    batch_size: int,
    num_electrons: int = 2,
    num_nuclei: int = 2,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "SerializedSpinSystem":
    """High-level factory for a complete SerializedSpinSystem."""
    num_hfc = min(num_electrons, num_nuclei)
    num_zfs = num_electrons
    num_dip = 1 if num_electrons > 1 else 0
    num_nn = num_nuclei - 1 if num_nuclei > 1 else 0
    
    return SerializedSpinSystem(
        metadata=make_spin_system_metadata(num_electrons, num_nuclei),
        interactions=make_spin_system_interactions(
            batch_size, num_electrons, num_hfc, num_zfs, num_dip, num_nn, device, dtype
        ),
        strain=make_spin_system_strains(
            batch_size, num_electrons, num_hfc, num_zfs, num_dip, device, dtype
        )
    )

In [4]:
def make_sample_width(
    batch_size: int,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "SerializedSampleWidth":
    return SerializedSampleWidth(
        ham_strain=torch.randn(batch_size, 3, device=device, dtype=dtype),
        gauss=torch.randn(batch_size, device=device, dtype=dtype),
        lorentz=torch.randn(batch_size, device=device, dtype=dtype)
    )

def make_sample_metadata(sample_type: str = "MultiOrientedSample") -> "SampleMetaData":
    mesh_meta = {"type": "DelaunayMesh", "some_param": 1.0}
    return SampleMetaData(sample_type=sample_type, mesh_meta=mesh_meta)

def make_serialized_sample(
    batch_size: int,
    num_electrons: int = 2,
    num_nuclei: int = 2,
    sample_type: str = "MultiOrientedSample",
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "SerializedSample":
    return SerializedSample(
        metadata=make_sample_metadata(sample_type),
        serialized_spin_system=make_serialized_spin_system(batch_size, num_electrons, num_nuclei, device, dtype),
        width=make_sample_width(batch_size, device=device, dtype=dtype)
    )

def make_serialized_mars_session(
    batch_size: int,
    num_electrons: int = 2,
    num_nuclei: int = 2,
    num_spectral_points: int = 500,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "SerializedMarsSession":
    return SerializedMarsSession(
        sample=make_serialized_sample(batch_size, num_electrons, num_nuclei, device=device, dtype=dtype),
        experimental_parameters=make_experimental_parameters(batch_size, num_spectral_points, include_time=True, device=device, dtype=dtype),
        polarization=make_polarization_parameters(batch_size, device=device, dtype=dtype)
    )

In [5]:
def make_polarization_parameters(
    batch_size: int,
    num_states: int = 10,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "PolarizationParameters":
    return PolarizationParameters(
        temperature=torch.full((batch_size,), 300.0, device=device, dtype=dtype),
        basis="Z",
        initial_populations=torch.randn(batch_size, num_states, device=device, dtype=dtype)
    )

def make_time_parameters(
    batch_size: int,
    num_points: int = 100,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "TimeParameters":
    return TimeParameters(
        min_time=torch.zeros(batch_size, device=device, dtype=dtype),
        max_time=torch.full((batch_size,), 1e-6, device=device, dtype=dtype),
        num_points=num_points
    )

def make_experimental_parameters(
    batch_size: int,
    num_points: int = 500,
    include_time: bool = False,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "ExperimentalParameters":
    time_params = make_time_parameters(batch_size, num_points=50, device=device, dtype=dtype) if include_time else None
    
    return ExperimentalParameters(
        min_pos=torch.full((batch_size,), 300.0, device=device, dtype=dtype),
        max_pos=torch.full((batch_size,), 400.0, device=device, dtype=dtype),
        num_points=num_points,
        resonance_parameter=torch.full((batch_size,), 9.5e9, device=device, dtype=dtype),
        time_params=time_params
    )

def make_cw_spectral_data(
    batch_size: int,
    num_points: int = 500,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "CWSpectralData":
    return CWSpectralData(
        experimental_parameters=make_experimental_parameters(batch_size, num_points, device=device, dtype=dtype),
        spectrum=torch.randn(batch_size, num_points, device=device, dtype=dtype)
    )

In [6]:
def make_graph_spin_system(
    batch_size: int,
    num_electrons: int = 2,
    num_nuclei: int = 2,
    num_interactions: int = 6,
    num_edges: int = 20,
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "GraphSpinSystem":
    num_nodes = num_electrons + num_nuclei + num_interactions
    
    source = torch.randint(0, num_nodes, (num_edges,), device=device)
    destination = torch.randint(0, num_nodes, (num_edges,), device=device)
    
    spins_1d = [0.5] * num_electrons + [1.0 if i % 2 == 0 else 0.5 for i in range(num_nuclei)] + [0.0] * num_interactions
    spins = torch.tensor(spins_1d, device=device, dtype=dtype).unsqueeze(0).expand(batch_size, -1)
    
    types_1d = [0] * num_electrons + [1] * num_nuclei + [2] * num_interactions
    node_types = torch.tensor(types_1d, device=device, dtype=torch.long).unsqueeze(0).expand(batch_size, -1)
    
    return GraphSpinSystem(
        source=source,
        destination=destination,
        components=torch.randn(batch_size, num_nodes, 3, device=device, dtype=dtype),
        angles=torch.randn(batch_size, num_nodes, 3, device=device, dtype=dtype),
        spins=spins,
        node_types=node_types
    )

def make_graph_sample(
    batch_size: int,
    num_electrons: int = 2,
    num_nuclei: int = 2,
    num_interactions: int = 6,
    num_edges: int = 20,
    sample_type: str = "MultiOrientedSample",
    device: torch.device = torch.device('cpu'),
    dtype: torch.dtype = torch.float32
) -> "GraphSample":
    return GraphSample(
        metadata=make_sample_metadata(sample_type),
        graph_spin_system=make_graph_spin_system(
            batch_size, num_electrons, num_nuclei, num_interactions, num_edges, device, dtype
        ),
        width=make_sample_width(batch_size, device=device, dtype=dtype)
    )

In [7]:
BATCH_SIZE = 4
NUM_ELECTRONS = 2
NUM_NUCLEI = 3
DEVICE = torch.device('cpu')
DTYPE = torch.float32

session = make_serialized_mars_session(
    batch_size=BATCH_SIZE, 
    num_electrons=NUM_ELECTRONS, 
    num_nuclei=NUM_NUCLEI,
    device=DEVICE, 
    dtype=DTYPE
)

graph_samp = make_graph_sample(
    batch_size=BATCH_SIZE,
    num_electrons=NUM_ELECTRONS,
    num_nuclei=NUM_NUCLEI,
    device=DEVICE,
    dtype=DTYPE
)


serialized_from_graph = graph_samp.to_serialized_sample()
graph_from_serialized = GraphSample.from_serialized_sample(session.sample)



### 2. Test conversion between representations.

In [8]:
sample = session.sample

graph_from_serialized = GraphSample.from_serialized_sample(session.sample)
double_conversion = graph_from_serialized.to_serialized_sample()


In [18]:
import torch

def compare_dataclasses(obj1, obj2):
    inter1 = obj1.interactions
    inter2 = obj2.interactions
    inter1 = dataclasses.asdict(inter1)
    inter2 = dataclasses.asdict(inter2)
    
    assert inter1.keys() == inter2.keys(),"keys of dataclsess are different"
    
    for key in inter1.keys():
        value_1 = inter1[key]
        value_2 = inter2[key]
        if value_1 is None:
            if value_2 is not None:
                print(f"Both values must be none at {key}")
            else:
                continue
        assert torch.allclose(value_1, value_2), f"tensors in the {key} are differnt"
        return True
    

def compare_spin_systems(obj1, obj2):
    """
    Recursively compares two SerializedSpinSystem objects (or their nested fields).
    Handles PyTorch tensors, lists, tuples, None, and nested objects.
    """
    return compare_dataclasses(obj1, obj2)



is_identical = compare_spin_systems(double_conversion.serialized_spin_system, sample.serialized_spin_system)

if is_identical:
    print("\n✅ Success: Both spin systems are identical!")
else:
    print("\n❌ Failure: The spin systems have differences (see logs above).")


✅ Success: Both spin systems are identical!
